In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [64]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# **STEP 1 : Indexing**
- Indexing in RAG means organizing and preparing our data so the system can quickly find the most relevant information when a user asks a question.
- It usually involves 4 steps to make the knowledge base searchable and efficient.
    - (a) Document Ingestion
    - (b) Text Chunking 
    - (c) Embedding Generation
    - (d) Storage in a Vector Store

---

### 1(a) Document Ingestion


In [72]:
video_id = "swCPic00c30" # Replace with your YouTube video ID (only the ID, not the full URL)

try:
    ytt_api = YouTubeTranscriptApi()
    # If you don't care which language, this returns the 'best' one
    transcript_list = ytt_api.fetch(video_id)


    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)


except TranscriptsDisabled:
    print("No captions available for this video.")
    

hello all my name is krishak and welcome to my YouTube channel so guys here is one amazing one short video on Lang chain in order to learn generative AI so if you are interested in creating amazing llm application or gen AI power uh application then this specific video is definitely for you if you don't know about Lang chain it is a complete framework that will actually help you to create Q&A chatbots rag application and many more the most interesting thing will be that I will be covering all the paid llm models along with that open llm models even though that they are hosted in hunging face so we will be getting to know each and everything about it and how we can specifically use in Lang chain so I hope you enjoy this particular video and please make sure that you watch this video till the end so thank you let's go ahead and enjoy the series uh before I go ahead guys uh since uh you know there's some there should be some motivation for me also so I will keep the light Target to th000 

In [25]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hello all my name is krishak and welcome', start=0.52, duration=3.759), FetchedTranscriptSnippet(text='to my YouTube channel so guys here is', start=2.44, duration=4.319), FetchedTranscriptSnippet(text='one amazing one short video on Lang', start=4.279, duration=5.121), FetchedTranscriptSnippet(text='chain in order to learn generative AI so', start=6.759, duration=4.081), FetchedTranscriptSnippet(text='if you are interested in creating', start=9.4, duration=4.439), FetchedTranscriptSnippet(text='amazing llm application or gen AI power', start=10.84, duration=5.6), FetchedTranscriptSnippet(text='uh application then this specific video', start=13.839, duration=4.561), FetchedTranscriptSnippet(text="is definitely for you if you don't know", start=16.44, duration=3.28), FetchedTranscriptSnippet(text='about Lang chain it is a complete', start=18.4, duration=3.0), FetchedTranscriptSnippet(text='framework that will actually help you t

### 1(b) Text Splitting

In [26]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [27]:
len(chunks)

221

In [28]:
chunks[0]

Document(metadata={}, page_content="hello all my name is krishak and welcome to my YouTube channel so guys here is one amazing one short video on Lang chain in order to learn generative AI so if you are interested in creating amazing llm application or gen AI power uh application then this specific video is definitely for you if you don't know about Lang chain it is a complete framework that will actually help you to create Q&A chatbots rag application and many more the most interesting thing will be that I will be covering all the paid llm models along with that open llm models even though that they are hosted in hunging face so we will be getting to know each and everything about it and how we can specifically use in Lang chain so I hope you enjoy this particular video and please make sure that you watch this video till the end so thank you let's go ahead and enjoy the series uh before I go ahead guys uh since uh you know there's some there should be some motivation for me also so I 

In [29]:
chunks[0].page_content

"hello all my name is krishak and welcome to my YouTube channel so guys here is one amazing one short video on Lang chain in order to learn generative AI so if you are interested in creating amazing llm application or gen AI power uh application then this specific video is definitely for you if you don't know about Lang chain it is a complete framework that will actually help you to create Q&A chatbots rag application and many more the most interesting thing will be that I will be covering all the paid llm models along with that open llm models even though that they are hosted in hunging face so we will be getting to know each and everything about it and how we can specifically use in Lang chain so I hope you enjoy this particular video and please make sure that you watch this video till the end so thank you let's go ahead and enjoy the series uh before I go ahead guys uh since uh you know there's some there should be some motivation for me also so I will keep the light Target to th000

### 1(c) Embedding Generation 

In [30]:
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

In [31]:
embeddings 

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x0000025BF420D8B0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000025BF48BFF50>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

### 1(d) Storing in Vector Store

In [32]:
vector_store = FAISS.from_documents(chunks,embeddings)

In [33]:
vector_store.index_to_docstore_id

{0: 'dd503054-039c-4750-be22-dc3b754d1b2b',
 1: '056432d5-9241-43d7-8326-953b508e5cd4',
 2: '44990787-ebff-4af7-8cc9-2f448d25a38b',
 3: '8ed11de7-1f6c-4fd3-a048-3b8be47e4482',
 4: '40b5242b-8015-483e-926d-5a1ba12575ca',
 5: 'f7c630ec-dc87-4b44-95c7-d4d7cf6f9b57',
 6: '3635d619-d0fa-4a0f-aa8d-11fba1769ce9',
 7: '4dd3dff3-72fe-46e2-b449-73098e14193a',
 8: 'bdf70948-2ea6-4007-913f-30cf19a18ab0',
 9: '121827a2-e59b-4ffd-bbac-ad3691afbca3',
 10: '002418bf-cb9d-4cb7-8116-6e5b48963e44',
 11: '837aebd3-1fb1-42a3-a422-446b1de5dc14',
 12: 'f96a9759-0c1c-445f-81a6-6e54cfbccc6c',
 13: '091c332c-8b8c-471f-bf15-a384d0e822af',
 14: '0c29821a-30a0-4a85-a145-e291a447c18e',
 15: 'e22ff715-6444-4b46-8ce6-3788503aa491',
 16: 'cc589a43-ab32-455f-9376-785e33da6256',
 17: '3c5ddb5c-a714-491e-80a1-035251e37055',
 18: 'f105505c-6009-4b20-b707-774333259091',
 19: '002caf39-ba51-47e1-a641-32d5c2874675',
 20: '7bbc0287-3dc4-4d24-823f-d1d9ee8d1ff6',
 21: 'b4318a29-dd93-4eee-b3bc-fb37cfc92f61',
 22: 'cbab913d-7f95-

In [34]:
vector_store.get_by_ids(['dd503054-039c-4750-be22-dc3b754d1b2b'])

[Document(id='dd503054-039c-4750-be22-dc3b754d1b2b', metadata={}, page_content="hello all my name is krishak and welcome to my YouTube channel so guys here is one amazing one short video on Lang chain in order to learn generative AI so if you are interested in creating amazing llm application or gen AI power uh application then this specific video is definitely for you if you don't know about Lang chain it is a complete framework that will actually help you to create Q&A chatbots rag application and many more the most interesting thing will be that I will be covering all the paid llm models along with that open llm models even though that they are hosted in hunging face so we will be getting to know each and everything about it and how we can specifically use in Lang chain so I hope you enjoy this particular video and please make sure that you watch this video till the end so thank you let's go ahead and enjoy the series uh before I go ahead guys uh since uh you know there's some there

---
# **STEP 2 : Retriever**
- Retrieval is the process of quickly searching the indexed knowledge base to get the most relevant information related to the user’s query.


In [39]:
# retriever = retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20
    }
)

In [40]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025BF49499D0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 20})

In [41]:
retriever.invoke("What is LangChain?")

[Document(id='40b5242b-8015-483e-926d-5a1ba12575ca', metadata={}, page_content="with the help of flask or some other some other libraries but here Lang uses something called as fast API okay and because of this fast API you know creation of this particular apis becomes very much easy so we'll be also able to understand it before the deployment if I have actually created my own llm app how I can actually create all the services in the form of apis that we'll try to see with respect to the Langer now coming to the next thing there are some amazing Concepts and important Concepts in Lang chain from data inje to data transformation and all in that major major topics are with respect to chains we'll try to understand about chains and probably in the next video once I probably start the Practical implementation the first thing that I'm actually going to cover is with respect to chains and I'm also going to discuss something called as agents and retrieval right not only that U you'll be able 

---
# **STEP 3 : Augmentation**
- Augmentation is the process of adding the retrieved relevant information to the user’s question so the LLM gets a richer and more informative prompt to generate a better answer.


In [54]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [55]:
prompt = PromptTemplate(
    template = """
               You are a helpful assistant.
               Answer only from the provided transcript context.
               If the context is insufficient to answer the question, say you don't know.

               {context}
               Question: {question}
               """,
    input_variables=["context", "question"]
)

In [56]:
question = "What is discussed in this video?"
retrieved_docs = retriever.invoke(question)

In [57]:
retrieved_docs

[Document(id='091c332c-8b8c-471f-bf15-a384d0e822af', metadata={}, page_content="many more things is going to come up so please make sure that we'll keep a like Target for every video and for this video the like Target is 1,000 and at least 200 comments and please make sure that you watch this video till the end because it is going to be completely practical oriented okay and uh if you really want to support please make sure that you subscribe the channel and take up membership plan from my YouTube channel so that it will help me and with the help of those benefits I will be able to create more videos as such so let me quickly go ahead and share my screen so here is my screen over here and you'll be able to see in the GitHub that you'll be finding in the description of this particular video you'll be having folders like this so today is the third tutorial not third second tutorial uh in the first and second we just understood that what all things we are going to learn but in this is the

In [58]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [59]:
context_text

"many more things is going to come up so please make sure that we'll keep a like Target for every video and for this video the like Target is 1,000 and at least 200 comments and please make sure that you watch this video till the end because it is going to be completely practical oriented okay and uh if you really want to support please make sure that you subscribe the channel and take up membership plan from my YouTube channel so that it will help me and with the help of those benefits I will be able to create more videos as such so let me quickly go ahead and share my screen so here is my screen over here and you'll be able to see in the GitHub that you'll be finding in the description of this particular video you'll be having folders like this so today is the third tutorial not third second tutorial uh in the first and second we just understood that what all things we are going to learn but in this is the real practical implementation that is probably there so as usual the first\n\n

In [60]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [61]:
final_prompt

StringPromptValue(text="\n               You are a helpful assistant.\n               Answer only from the provided transcript context.\n               If the context is insufficient to answer the question, say you don't know.\n\n               many more things is going to come up so please make sure that we'll keep a like Target for every video and for this video the like Target is 1,000 and at least 200 comments and please make sure that you watch this video till the end because it is going to be completely practical oriented okay and uh if you really want to support please make sure that you subscribe the channel and take up membership plan from my YouTube channel so that it will help me and with the help of those benefits I will be able to create more videos as such so let me quickly go ahead and share my screen so here is my screen over here and you'll be able to see in the GitHub that you'll be finding in the description of this particular video you'll be having folders like this

---
# **STEP 4 : Generation**
- Generation is the final stage where the LLM uses the user’s question along with the retrieved and enriched context to create a meaningful response.


In [62]:
answer = llm.invoke(final_prompt)
print(answer.content)

The video discusses the practical implementation of loading, transforming, and embedding data, specifically in the context of using large language models (LLMs). It covers the process of reading data from a specific source, performing feature engineering, breaking data into smaller chunks for context size considerations, and converting those chunks into vectors. The video also introduces the use of FastAPI for creating APIs, discusses important concepts in LangChain such as chains and agents, and outlines the steps for creating a Q&A application using open-source LLM models. Additionally, it mentions the use of different vector databases and provides a GitHub link for resources related to the tutorial.


# **STEP 5 : Building a Chain**

In [65]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [66]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [67]:
parallel_chain.invoke("What is LangChain?")

{'context': "with the help of flask or some other some other libraries but here Lang uses something called as fast API okay and because of this fast API you know creation of this particular apis becomes very much easy so we'll be also able to understand it before the deployment if I have actually created my own llm app how I can actually create all the services in the form of apis that we'll try to see with respect to the Langer now coming to the next thing there are some amazing Concepts and important Concepts in Lang chain from data inje to data transformation and all in that major major topics are with respect to chains we'll try to understand about chains and probably in the next video once I probably start the Practical implementation the first thing that I'm actually going to cover is with respect to chains and I'm also going to discuss something called as agents and retrieval right not only that U you'll be able to see there is a concept of LC okay now in this LC we will be disc

In [68]:
parser = StrOutputParser()

In [69]:
main_chain = parallel_chain | prompt | llm | parser

In [70]:
main_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025BF49499D0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 20})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n               You are a helpful assistant.\n               Answer only from the provided transcript context.\n               If the context is insufficient to answer the question, say you don't know.\n\n               {context}\n               Question: {question}\n               ")
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False

In [71]:
main_chain.invoke("Can You summarize the video?")

'The video focuses on learning about LangChain, its ecosystem, and how to work with documents using vector embedding techniques. The presenter sets a target for likes and comments to motivate viewers and discusses recent updates in LangChain documentation. They explain the process of dividing a PDF document into smaller chunks and converting those into vectors. The video will include practical implementation using VS Code, where the presenter will create a Jupyter notebook file for a project. They plan to use open-source LLM models and demonstrate building a Q&A application with data loaded from PDFs. The presenter emphasizes the importance of learning and encourages viewer feedback.'

In [ ]:
import streamlit as st
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from urllib.parse import urlparse, parse_qs
import re

# Load environment variables
load_dotenv()

# Function to extract video ID from YouTube URL
def extract_video_id(url):
    """
    Extract video ID from various YouTube URL formats
    Supports:
    - https://www.youtube.com/watch?v=VIDEO_ID
    - https://youtu.be/VIDEO_ID
    - https://www.youtube.com/watch?v=VIDEO_ID&other_params
    - Just the video ID itself
    """
    # If it's just a video ID (11 characters, alphanumeric and some special chars)
    if re.match(r'^[a-zA-Z0-9_-]{11}$', url.strip()):
        return url.strip()
    
    try:
        # Handle youtu.be short links
        if 'youtu.be' in url:
            return url.split('/')[-1].split('?')[0]
        
        # Handle youtube.com links
        if 'youtube.com' in url:
            parsed_url = urlparse(url)
            video_id = parse_qs(parsed_url.query).get('v', [None])[0]
            if video_id:
                return video_id
        
        # If it looks like a URL but doesn't match above patterns
        return None
    except:
        return None

# Set page config
st.set_page_config(page_title="YouTube Chatbot - RAG", layout="wide")
st.title("🎬 YouTube Video Q&A Chatbot")
st.markdown("---")

# Initialize session state
if "vector_store" not in st.session_state:
    st.session_state.vector_store = None
if "main_chain" not in st.session_state:
    st.session_state.main_chain = None
if "video_id" not in st.session_state:
    st.session_state.video_id = None
if "transcript" not in st.session_state:
    st.session_state.transcript = None
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# Sidebar for video input
st.sidebar.header("📹 Video Setup")
video_input = st.sidebar.text_input(
    "",
    placeholder="Enter YouTube URL or Video ID",
    help="Paste the full YouTube URL or just the video ID"
)

if st.sidebar.button("🔍 Load Video & Build RAG", key="load_video"):
    if not video_input.strip():
        st.error("Please enter a YouTube URL or Video ID")
    else:
        # Extract video ID from URL or use it directly
        video_id = extract_video_id(video_input)
        
        if not video_id:
            st.error("❌ Invalid YouTube URL or Video ID. Please check and try again.")
            st.stop()
        
        with st.spinner("Fetching transcript..."):
            try:
                # Step 1(a): Document Ingestion
                ytt_api = YouTubeTranscriptApi()
                
                # Try to fetch transcripts in multiple languages
                # Priority: English, Hindi, then any available language
                transcript_list = None
                language_used = None
                
                try:
                    # Try English first
                    transcript_list = ytt_api.fetch(video_id, languages=['en'])
                    language_used = "English"
                except:
                    try:
                        # Try Hindi
                        transcript_list = ytt_api.fetch(video_id, languages=['hi'])
                        language_used = "Hindi"
                    except:
                        try:
                            # Try to get any available language
                            transcript_list = ytt_api.fetch(video_id)
                            language_used = "Available"
                        except:
                            transcript_list = None
                
                if transcript_list is None:
                    raise TranscriptsDisabled("No transcripts available")
                
                transcript = " ".join(chunk.text for chunk in transcript_list)
                
                st.session_state.transcript = transcript
                st.session_state.video_id = video_id
                st.session_state.chat_history = []
                
                st.sidebar.success("✅ Transcript fetched successfully!")
                
            except TranscriptsDisabled:
                st.error("❌ No captions available for this video. Please try another video.")
                st.stop()
            except Exception as e:
                st.error(f"❌ Error fetching transcript: {str(e)}")
                st.stop()

        # Step 1(b): Text Splitting
        with st.spinner("Chunking text..."):
            splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
            chunks = splitter.create_documents([transcript])

        # Step 1(c) & (d): Embedding Generation & Vector Store
        with st.spinner("Creating embeddings and building vector store..."):
            embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
            vector_store = FAISS.from_documents(chunks, embeddings)
            st.session_state.vector_store = vector_store

        # Step 2: Retriever
        retriever = vector_store.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": 5,
                "fetch_k": 20
            }
        )

        # Step 3: Augmentation - Prompt Template
        # prompt = PromptTemplate(
        #     template="""
        #        You are a helpful assistant.
        #        Answer only from the provided transcript context.
        #        If the context is insufficient to answer the question, say you don't know.

        #        {context}
        #        Question: {question}
        #        """,
        #     input_variables=["context", "question"]
        # )


        prompt = PromptTemplate(
    template="""
                You are a precise and helpful assistant for YouTube video Q&A.

                Rules:
                1. Answer using ONLY the provided transcript context.
                2. Do not use outside knowledge or assumptions.
                3. If the context is missing, unclear, or insufficient, reply exactly:
                "I don't know based on the provided transcript."
                4. Keep the answer concise and accurate.
                5. If the user asks in Hindi, reply in Hindi. Otherwise, reply in the same language as the question.
                6. When possible, include short supporting lines/phrases from the context in your answer.

                Transcript Context:
                {context}

                Question:
                {question}

                Answer:
                """,
    input_variables=["context", "question"]
)

        # Step 4: LLM Setup
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

        # Step 5: Building a Chain
        def format_docs(retrieved_docs):
            context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
            return context_text

        parallel_chain = RunnableParallel({
            'context': retriever | RunnableLambda(format_docs),
            'question': RunnablePassthrough()
        })

        parser = StrOutputParser()
        main_chain = parallel_chain | prompt | llm | parser

        st.session_state.main_chain = main_chain
        st.success("✅ RAG System Ready!")

# Main content area
if st.session_state.video_id:
    st.info(f"📌 Loaded video ID: `{st.session_state.video_id}`")
    st.markdown("---")

    # Display transcript info
    col1, col2 = st.columns(2)
    with col1:
        if st.button("📄 Show Transcript"):
            with st.expander("Full Transcript"):
                st.text_area("Transcript Content", value=st.session_state.transcript, height=300, disabled=True)
    
    with col2:
        if st.button("🔤 Show Chunks Info"):
            if st.session_state.vector_store:
                with st.expander("Chunking Information"):
                    st.write(f"**Total Chunks:** {len(st.session_state.vector_store.index_to_docstore_id)}")
                    st.write("**Chunk Configuration:**")
                    st.write("- Chunk Size: 1000")
                    st.write("- Chunk Overlap: 200")
                    st.write("- Search Type: MMR (Maximal Marginal Relevance)")
                    st.write("- Top K Results: 5")

    st.markdown("---")
    st.header("💬 Chat with Video")
    
    # Display chat history
    st.write("")
    for message in st.session_state.chat_history:
        if message["role"] == "user":
            st.chat_message("user").write(message["content"])
        else:
            st.chat_message("assistant").write(message["content"])
    
    st.write("")
    
    # Chat input
    question = st.chat_input("Ask a question about the video...")
    
    if question:
        if st.session_state.main_chain is None:
            st.error("Please load a video first")
        else:
            # Add user message to history
            st.session_state.chat_history.append({"role": "user", "content": question})
            
            # Display user message
            st.chat_message("user").write(question)
            
            # Generate answer
            with st.spinner("Thinking..."):
                try:
                    answer = st.session_state.main_chain.invoke(question)
                    
                    # Add assistant message to history
                    st.session_state.chat_history.append({"role": "assistant", "content": answer})
                    
                    # Display assistant message
                    st.chat_message("assistant").write(answer)
                    
                except Exception as e:
                    st.error(f"Error generating answer: {str(e)}")

else:
    st.info("👈 Please enter a YouTube URL or Video ID in the sidebar to get started!")
    
    st.markdown("---")
    st.subheader("ℹ️ How to use:")
    st.markdown("""
    1. **Enter YouTube URL**: Paste the full YouTube URL or just the Video ID in the sidebar
    2. **Load Video**: Click the "Load Video & Build RAG" button
    3. **Start Chatting**: Once the system is ready, start asking questions about the video
    4. **View Transcript**: Optionally view the full transcript or chunking information
    
    **Supported Languages**: English 🇬🇧 | Hindi 🇮🇳 | And any other available language
    
    The system will:
    - Extract the video transcript (in any available language)
    - Split it into manageable chunks
    - Create embeddings for semantic search
    - Build a retriever to find relevant sections
    - Generate answers based on the video content in a ChatGPT-like interface
    """)

    st.markdown("---")
    st.subheader("📚 Example:")
    st.code("""
    Full URL: https://www.youtube.com/watch?v=swCPic00c30
    or Short URL: https://youtu.be/swCPic00c30
    or Just ID: swCPic00c30
    
    Questions you can ask:
    - What is discussed in this video?
    - Can you summarize the video?
    - What is LangChain?
    """)
